# create_estdata.ipynb

This notebook creates the input parquet needed for the US logit model. Starting from `pums/pums_{year}.parquet` (built by `pums/unify_pums.ipynb`, which already has `STAY`/`ORIGIN`/`CHOSEN`), it merges in origin-side (MIGPUMA) census (ACS+LODES) and CBSA data, draws `num_alternatives` random destination-PUMA alternatives per person (always including the true `CHOSEN` PUMA for movers), and fills each alternative with census/CBSA/distance/travel-time/own-industry-job data.

In [1]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))
from lib import io as lio

In [2]:
year = 2018
num_alternatives = 200

In [3]:
pums = pd.read_parquet(f"pums/pums_{year}.parquet")
pums

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,ST,ADJINC,AGEP,CIT,...,IN_MILITARY,UNEMPLOYED,NOT_IN_LABOR_FORCE,IN_LABOR_FORCE,WHITE,BLACK,INDIAN,AAPI,LATINO,RACE_ETHNICITY
0,P,2018HU0637969,3,1,1104,2,17,1013097,61,1,...,0,0,0,1,1,0,0,0,0,1
1,P,2018HU0808038,5,5,505,3,24,1013097,70,4,...,0,0,1,0,0,0,0,1,0,6
2,P,2018HU1051188,9,1,7310,4,6,1013097,37,1,...,0,0,0,1,1,0,0,0,0,1
3,P,2018HU0461416,2,1,4110,1,36,1013097,55,4,...,0,0,0,1,1,0,0,0,0,1
4,P,2018GQ0016994,5,1,51167,3,51,1013097,40,1,...,0,0,1,0,0,1,0,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253085,P,2018HU0435287,5,2,1400,3,24,1013097,45,1,...,0,0,0,1,1,0,0,0,0,1
253086,P,2018HU0652247,8,2,900,4,8,1013097,23,1,...,0,1,0,1,1,0,0,0,0,1
253087,P,2018HU0192642,8,1,824,4,8,1013097,57,1,...,0,0,0,1,1,0,0,0,0,1
253088,P,2018HU0750302,7,1,5915,3,48,1013097,54,1,...,0,0,0,1,1,0,0,0,0,1


In [4]:
# load the combined ACS+LODES census extract and the CBSA density/type extract,
# for both PUMA (destination/alternative side) and MIGPUMA (origin side) geographies.
# PUMA/MIGPUMA/GEOID/GISMATCH all round-trip through csv as unpadded ints, so re-zfill them.
puma_census = (
    pd.read_csv(f"census/puma_{year}.csv", dtype={"PUMA": str})
    .assign(PUMA=lambda d: d["PUMA"].str.zfill(7))
    .set_index("PUMA")
)
migpuma_census = (
    pd.read_csv(f"census/migpuma_{year}.csv", dtype={"MIGPUMA": str})
    .assign(MIGPUMA=lambda d: d["MIGPUMA"].str.zfill(7))
    .set_index("MIGPUMA")
)

puma_cbsa = (
    pd.read_csv(f"geometry/cbsa/puma_density_{year}.csv", dtype={"GEOID": str})
    .assign(GEOID=lambda d: d["GEOID"].str.zfill(7))
    .set_index("GEOID")
)
migpuma_cbsa = (
    pd.read_csv(f"geometry/cbsa/migpuma_density_{year}.csv", dtype={"GISMATCH": str})
    .assign(GISMATCH=lambda d: d["GISMATCH"].str.zfill(7))
    .set_index("GISMATCH")
)

# PUMA<->MIGPUMA equivalency table (already indexed by PUMA, State already zero-padded
# as it's read as dtype=str) -- gives us each alternative PUMA's state, for ALTi_STATE
puma_migpuma = lio.load_puma_migpuma("geometry/equivalencies/puma_migpuma.csv")

In [5]:
# encode CBSA type (T34/Metro/Micro/other) numerically, and build a single CBSA-name
# codebook shared between the PUMA and MIGPUMA extracts (factorized on the PUMA side,
# since alternatives are PUMA-keyed) so ALTi_CBSA and CBSA_ORIG are directly comparable.
def encode_cbsa_type(df):
    out = np.full(len(df), 3)
    out = np.where(df["TYPE"] == "Micro", 2, out)
    out = np.where(df["TYPE"] == "Metro", 1, out)
    out = np.where(df["TYPE"] == "T34", 0, out)
    out = np.where(df["TYPE"] == "NOT_CBSA", -2, out)
    df["TYPE_NUM"] = out
    return df


puma_cbsa = encode_cbsa_type(puma_cbsa)
migpuma_cbsa = encode_cbsa_type(migpuma_cbsa)

puma_cbsa["NAME_NUM"], cbsa_name_categories = pd.factorize(puma_cbsa["CBSA_NAME"])
cbsa_name_to_num = dict(zip(cbsa_name_categories, range(len(cbsa_name_categories))))
migpuma_cbsa["NAME_NUM"] = (
    migpuma_cbsa["CBSA_NAME"].map(cbsa_name_to_num).fillna(-2).astype(int)
)

In [6]:
df = pums.merge(
    migpuma_census.add_suffix(
        ".ORIG"
    ),  # full curated+uncurated ACS/LODES census columns
    how="left",
    left_on="ORIGIN",
    right_index=True,  # migpuma_census is indexed by MIGPUMA, matched against ORIGIN
).merge(
    # only pull the handful of CBSA columns actually needed, not the whole table
    migpuma_cbsa[["CBSA_NAME", "NAME_NUM", "TYPE", "TYPE_NUM"]].add_suffix(".ORIG"),
    how="left",
    left_on="ORIGIN",
    right_index=True,  # migpuma_cbsa is indexed by GISMATCH, matched against ORIGIN
)
df

,RT,SERIALNO,DIVISION,SPORDER,PUMA,REGION,ST,ADJINC,AGEP,CIT,...,Proportion of other services jobs.ORIG,Proportion of public administration jobs.ORIG,Proportion of jobs with less than high school education.ORIG,"Proportion of jobs with high school education, no college.ORIG",Proportion of jobs with some college or associate degree.ORIG,Proportion of jobs with bachelor's degree or advanced degree.ORIG,CBSA_NAME.ORIG,NAME_NUM.ORIG,TYPE.ORIG,TYPE_NUM.ORIG
0,P,2018HU0637969,3,1,1104,2,17,1013097,61,1,...,0.043706,0.034705,0.119157,0.311076,0.339797,0.229971,"St. Louis, MO-IL Metro Area",218,T34,0
1,P,2018HU0808038,5,5,505,3,24,1013097,70,4,...,0.033583,0.061243,0.115287,0.261863,0.302088,0.320763,"Baltimore-Columbia-Towson, MD Metro Area",341,T34,0
2,P,2018HU1051188,9,1,7310,4,6,1013097,37,1,...,0.037008,0.031381,0.183056,0.206614,0.301505,0.308825,"San Diego-Carlsbad, CA Metro Area",13,T34,0
3,P,2018HU0461416,2,1,4110,1,36,1013097,55,4,...,0.037771,0.090732,0.171338,0.223698,0.283477,0.321488,"New York-Newark-Jersey City, NY-NJ-PA Metro Area",367,T34,0
4,P,2018GQ0016994,5,1,51167,3,51,1013097,40,1,...,0.037175,0.028129,0.129986,0.277694,0.319596,0.272724,"Virginia Beach-Norfolk-Newport News, VA-NC Met...",477,Metro,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253085,P,2018HU0435287,5,2,1400,3,24,1013097,45,1,...,0.034845,0.063407,0.131087,0.311471,0.315002,0.242440,"Salisbury, MD-DE Metro Area",321,Metro,1
253086,P,2018HU0652247,8,2,900,4,8,1013097,23,1,...,0.027779,0.098203,0.119982,0.295455,0.327084,0.257479,NaN,-2,NOT_CBSA,-2
253087,P,2018HU0192642,8,1,824,4,8,1013097,57,1,...,0.029676,0.049137,0.122259,0.236959,0.309498,0.331284,NaN,-2,NOT_CBSA,-2
253088,P,2018HU0750302,7,1,5915,3,48,1013097,54,1,...,0.028306,0.023055,0.202805,0.271469,0.320084,0.205643,"San Antonio-New Braunfels, TX Metro Area",89,T34,0


In [7]:
# map each person's raw NAICSP code to a NAICS sector abbreviation, needed for OWN_JOB
# below. ACS's NAICSP industry codes are structured so their first two characters
# already identify the standard 2-digit NAICS sector (e.g. "5411" -> sector 54,
# Professional Services), including two merged "not specified" codes ("3M"/"4M" for
# unspecified Manufacturing/Retail). This replaces the old per-year
# naics_to_lodes_{year}.txt crosswalk, which hand-maintained a label per detailed code
# and had at least one real bug (the 2017 file mislabeled all of NAICS 53 Real Estate as
# FIN instead of REL).
#
# NOTE: NAICS sector 92 (Public Administration) includes military
NAICS_SECTOR_PREFIXES = {
    "11": "AGR",
    "21": "EXT",
    "22": "UTL",
    "23": "CON",
    "31": "MFG",
    "32": "MFG",
    "33": "MFG",
    "3M": "MFG",
    "42": "WHL",
    "44": "RET",
    "45": "RET",
    "4M": "RET",
    "48": "TRN",
    "49": "TRN",
    "51": "INF",
    "52": "FIN",
    "53": "REL",
    "54": "PRF",
    "55": "MNG",
    "56": "ADM",
    "61": "EDU",
    "62": "MED",
    "71": "ENT",
    "72": "FOD",
    "81": "SRV",
    "92": "PUB",
    # "99" corresponds to other/unemployed
}

df["NAICS"] = df["NAICSP"].str[0:2].map(NAICS_SECTOR_PREFIXES)
# people in active service will be treated specially, exclude from PUB
df["NAICS"] = np.where(df["IN_MILITARY"], np.nan, df["NAICS"])
df["NAICS"].value_counts(dropna=False)

NAICS
NaN    69677
MED    24622
RET    20182
EDU    18644
MFG    18448
PRF    13539
FOD    12302
CON    11649
SRV     9247
PUB     9151
TRN     8246
FIN     8003
ADM     7542
WHL     4555
ENT     4386
INF     3562
REL     3540
AGR     2999
UTL     1601
EXT      895
MNG      300
Name: count, dtype: int64

In [8]:
df["NAICS_MED"] = np.where(df["NAICS"] == "MED", 1, 0)
df["NAICS_MFG"] = np.where(df["NAICS"] == "MFG", 1, 0)
df["NAICS_RET"] = np.where(df["NAICS"] == "RET", 1, 0)
df["NAICS_EDU"] = np.where(df["NAICS"] == "EDU", 1, 0)
df["NAICS_ADM"] = np.where(df["NAICS"] == "ADM", 1, 0)
df["NAICS_FOD"] = np.where(df["NAICS"] == "FOD", 1, 0)
df["NAICS_PRF"] = np.where(df["NAICS"] == "PRF", 1, 0)
df["NAICS_TRN"] = np.where(df["NAICS"] == "TRN", 1, 0)
df["NAICS_SRV"] = np.where(df["NAICS"] == "SRV", 1, 0)
df["NAICS_FIN"] = np.where(df["NAICS"] == "FIN", 1, 0)
df["NAICS_WHL"] = np.where(df["NAICS"] == "WHL", 1, 0)
df["NAICS_AGR"] = np.where(df["NAICS"] == "AGR", 1, 0)
df["NAICS_PUB"] = np.where(df["NAICS"] == "PUB", 1, 0)
df["NAICS_INF"] = np.where(df["NAICS"] == "INF", 1, 0)
df["NAICS_ENT"] = np.where(df["NAICS"] == "ENT", 1, 0)
df["NAICS_REL"] = np.where(df["NAICS"] == "REL", 1, 0)
df["NAICS_UTL"] = np.where(df["NAICS"] == "UTL", 1, 0)
df["NAICS_EXT"] = np.where(df["NAICS"] == "EXT", 1, 0)
df["NAICS_MNG"] = np.where(df["NAICS"] == "MNG", 1, 0)
df["NAICS_CON"] = np.where(df["NAICS"] == "CON", 1, 0)
df["NAICS_OTHER"] = np.where(df["NAICS"].isna(), 1, 0)

In [9]:
# combine NAICS job categories into groups, by skill/credential barrier to entry,
# for use in migration model. Shared by NAICS_{group} (this cell, individual-level
# one-hot dummies) and NAICS_GROUP_PCT_{group} (added onto puma_census/migpuma_census
# below, area-level job shares) so the two stay in sync.
NAICS_GROUPS = {
    # primary/extractive: resource-tied to location
    "AGR_EXT": ["AGR", "EXT"],
    # high-education professional: bachelor's+ degree typical, portable credentials
    "HIGH_ED": ["MED", "EDU", "PRF", "FIN", "INF", "MNG"],
    # occupational-license-heavy services: state-specific licenses (real estate,
    # personal care/repair, etc.), a classic migration friction
    "LICENSE": ["SRV", "REL"],
    # goods-producing/trade: apprenticeship or on-the-job skill, not degree/license driven
    "GOODS_TRADE": ["MFG", "CON", "WHL", "TRN", "UTL"],
    # low-skill consumer services: low-license, low-credential
    "LOW_SKILL_SVC": ["RET", "FOD", "ADM", "ENT"],
    # public administration, including active-duty military
    "GOVT": ["PUB"],
}

for group, sectors in NAICS_GROUPS.items():
    # max on [0, 1] is effectively an or
    df[f"NAICS_{group}"] = df[[f"NAICS_{s}" for s in sectors]].max(axis=1)

In [10]:
# current origin(MIGPUMA)->destination(PUMA) distance/time matrices (regenerated by
# distances/pums_distances.ipynb) -- rows=MIGPUMA, columns=PUMA, contiguous US only.
# columns give us the full alternative-PUMA pool (AK/HI/PR already excluded).
distance_matrix = lio.load_distance_matrix(
    "distances/puma_migpuma_distance_matrix.csv", zfill=7
)
time_matrix = lio.load_distance_matrix(
    "distances/puma_migpuma_time_matrix.csv", zfill=7
)
alt_pool = np.array(distance_matrix.columns, dtype="<U7")
len(alt_pool)

2336

In [11]:
# draw num_alternatives random destination PUMAs per person (vectorized, in row-batches to
# bound peak memory).
# For movers (STAY==0), slot 0 is forced to be their actual CHOSEN PUMA so the true choice
# is always present in the sampled choice set exactly once.
rng = np.random.default_rng(5063)  # fixed seed for reproducibility
n = len(df)  # number of PUMS records (rows) to sample alternatives for
pool_size = len(alt_pool)  # total number of candidate PUMAs to sample from

# output array: n rows x num_alternatives columns, each cell a PUMA code string (max 7 chars)
random_pumas = np.empty((n, num_alternatives), dtype="<U7")

chosen = df["CHOSEN"].to_numpy()  # each person's actual chosen destination PUMA
stay = df["STAY"].to_numpy()  # flag: 1 = stayed in place, 0 = moved to CHOSEN

# build a lookup from PUMA code -> its integer position within alt_pool, so we can
# work with the sampling in terms of array indices rather than string comparisons
pool_index = {puma: i for i, puma in enumerate(alt_pool)}

# for every record, find where its CHOSEN puma sits in alt_pool (-1 if not present,
# e.g. for stayers where CHOSEN may not be a valid/relevant destination)
chosen_pos = np.array([pool_index.get(p, -1) for p in chosen])

# sanity check: every mover's chosen destination must actually exist in the pool,
# otherwise we'd have no way to force it into slot 0 below
assert (chosen_pos[stay == 0] >= 0).all(), (
    "some movers' CHOSEN puma is missing from the alternative pool"
)

# process rows in batches so we never materialize an (n x pool_size) array,
# which could be huge for large PUMS extracts / large pools
batch_size = 25_000
for start in range(0, n, batch_size):
    end = min(start + batch_size, n)

    # draw one random uniform "key" per (row, pool-position) pair in this batch;
    # sorting these keys per row gives us a random permutation of pool positions
    # each of these are a value from 0-1
    keys = rng.random((end - start, pool_size))

    # argsort each row's keys and keep only the first num_alternatives columns:
    # this is equivalent to sampling num_alternatives distinct positions from the
    # pool without replacement, per row, without explicitly shuffling the whole pool
    # the argsort sorts the random values from 0-1, equivalent to a random permutation of all the possible pumas to move to
    # take the first num_alternatives of this permutation to get our desired alternatives
    order = np.argsort(keys, axis=1)[
        :, :num_alternatives
    ]  # random subset of pool positions, per row

    # slice this batch's chosen-position and mover-status arrays to match `order`
    batch_chosen_pos = chosen_pos[start:end]
    movers_mask = stay[start:end] == 0

    # for each row, check whether the CHOSEN position happens to already appear
    # somewhere among the num_alternatives sampled columns
    match = order == batch_chosen_pos[:, None]
    already_present = match.any(axis=1)
    # column index of the first (only, since positions are drawn w/o replacement) match;
    # meaningless for rows where already_present is False, but those get overwritten below anyway
    match_col = np.argmax(match, axis=1)

    # case 1: mover whose CHOSEN puma was NOT among the sampled alternatives ->
    # forcibly overwrite slot 0 with the chosen position, guaranteeing it's included
    need_insert = movers_mask & ~already_present
    order[need_insert, 0] = batch_chosen_pos[need_insert]

    # case 2: mover whose CHOSEN puma WAS sampled, but landed in some column other than 0 ->
    # swap it into slot 0 (rather than overwrite) so it still appears exactly once,
    # just relocated, preserving the rest of the sampled alternatives
    need_swap = movers_mask & already_present & (match_col != 0)
    rows_to_swap = np.nonzero(need_swap)[
        0
    ]  # row indices (within this batch) needing a swap
    cols_to_swap = match_col[need_swap]  # the column each of those rows' match sits in
    tmp = order[rows_to_swap, 0].copy()  # save whatever was in slot 0 before swapping
    order[rows_to_swap, 0] = order[
        rows_to_swap, cols_to_swap
    ]  # move chosen puma into slot 0
    order[rows_to_swap, cols_to_swap] = tmp  # put the displaced value in its place

    # note: stayers (movers_mask == False) are left untouched here — their slot 0
    # is whatever was randomly sampled, since there's no CHOSEN puma to force in

    # translate sampled pool positions back into actual PUMA code strings and store
    random_pumas[start:end] = alt_pool[order]

random_pumas.shape

(253090, 200)

In [12]:
# sanity checks on the drawn alternatives
movers = stay == 0
# make sure for all movers alt1 is the chosen
assert (random_pumas[movers, 0] == chosen[movers]).all(), (
    "ALT1 != CHOSEN for some movers"
)
sample_idx = rng.choice(n, size=min(500, n), replace=False)
dup_counts = np.array([len(set(row)) for row in random_pumas[sample_idx]])
assert (dup_counts == num_alternatives).all(), (
    "found a duplicate PUMA within a single person's alternative set"
)

In [13]:
# curated subset of ACS fields to bring in (verbose "Category.Subcategory.SE_CODE"
# column names resolved against the current census/puma_2018.csv & migpuma_2018.csv headers)
# NOTE: many of these are commented out since they add a significant amount of columns
CENSUS_FIELDS = {
    "TOT_POP": "Population Density (Per Sq. Mile).Total Population.SE_A00002_001",
    "DENS": "Population Density (Per Sq. Mile).Population Density (Per Sq. Mile).SE_A00002_002",
    # "POP_UND_18": "Age (Short Version).Total Population Under 18 Years.SE_B01001_002",
    # "POP_18_34": "Age (Short Version).Total Population 18 to 34 Years.SE_B01001_003",
    # "POP_35_64": "Age (Short Version).Total Population 35 to 64 Years.SE_B01001_004",
    # "POP_OVER_65": "Age (Short Version).Total Population 65 and Over.SE_B01001_005",
    "MED_AGE": "Median Age by Sex.Median Age.SE_A01004_001",
    "FOREIGN_BORN_PCT": "Proportion foreign born",
    "MIL_PCT": "Proportion of people in military",
    "HH_MED_INC": "Median Household Income (In 2018 Inflation Adjusted Dollars).Median Household Income (In 2018 Inflation Adjusted Dollars).SE_A14006_001",
    # "HU_TOT": "Housing Units.Housing Units.SE_A10001_001",
    # "HH_MED_VAL": "Median House Value for All Owner-Occupied Housing Units.Median Value.SE_A10036_001",
    "UNEMP_RATE": "Unemployment rate",
    "COLLEGE_PCT": "Proportion of people in college",
    "HOUSE_VACANCY_PCT": "House vacancy proportion",
    "MED_TRAVEL_TIME": "Median travel time",
    "MED_RENT_PCT_HH_INC": "Median gross rent as a percentage of household income",
    "MED_HOUSE_VALUE_OVER_MED_HH_INC": "Median house value over median household income",
    # "WHITE_PCT": "Proportion of People White",
    # "ASIAN_PCT": "Proportion of People Asian",
    # "PI_PCT": "Proportion of People PI",
    # "BLACK_PCT": "Proportion of People Black",
    # "LATINO_PCT": "Proportion of People Latino",
    # "INDIAN_PCT": "Proportion of People Indian",
    "POVERTY_PCT": "Proportion of people struggling",
    "YR_SINCE_MED_STRUCTURE": "Years since median structure built",
    "HH_WITH_CHILD_PCT": "Proportion of households with children",
    # "LF_PARTCP_RATE": "Labor force participation rate",
    "ALT_COMMUTE_PCT": "Proportion alternative commute",  # walking or public transit
    "MED_OWNER_COST_HH_INC_PCT": "Median selected monthly owner costs as percentage of household income",
    "ENT_JOBS_PCT": "Proportion of entertainment jobs",
    "TOT_JOBS": "Total number of jobs",
}
NAICS_SECTOR_COLUMNS = {
    "AGR": "Number of jobs in NAICS sector 11 (Agriculture, Forestry, Fishing and Hunting)",
    "EXT": "Number of jobs in NAICS sector 21 (Mining, Quarrying, and Oil and Gas Extraction)",
    "UTL": "Number of jobs in NAICS sector 22 (Utilities)",
    "CON": "Number of jobs in NAICS sector 23 (Construction)",
    "MFG": "Number of jobs in NAICS sector 31-33 (Manufacturing)",
    "WHL": "Number of jobs in NAICS sector 42 (Wholesale Trade)",
    "RET": "Number of jobs in NAICS sector 44-45 (Retail Trade)",
    "TRN": "Number of jobs in NAICS sector 48-49 (Transportation and Warehousing)",
    "INF": "Number of jobs in NAICS sector 51 (Information)",
    "FIN": "Number of jobs in NAICS sector 52 (Finance and Insurance)",
    "REL": "Number of jobs in NAICS sector 53 (Real Estate and Rental and Leasing)",
    "PRF": "Number of jobs in NAICS sector 54 (Professional, Scientific, and Technical Services)",
    "MNG": "Number of jobs in NAICS sector 55 (Management of Companies and Enterprises)",
    "ADM": "Number of jobs in NAICS sector 56 (Administrative and Support and Waste Management and Remediation Services)",
    "EDU": "Number of jobs in NAICS sector 61 (Educational Services)",
    "MED": "Number of jobs in NAICS sector 62 (Health Care and Social Assistance)",
    "ENT": "Number of jobs in NAICS sector 71 (Arts, Entertainment, and Recreation)",
    "FOD": "Number of jobs in NAICS sector 72 (Accommodation and Food Services)",
    "SRV": "Number of jobs in NAICS sector 81 (Other Services [except Public Administration])",
    "PUB": "Number of jobs in NAICS sector 92 (Public Administration)",
}

AGE_BRACKET_COLS = [
    "Proportion of people under 18",
    "Proportion of people 18-34",
    "Proportion of people 35-64",
    "Proportion of people 65+",
]


"""
RACE_ETHNICITY (derived from RAC1P)
Recoded detailed race code
1 .White alone
2 .Black or African American alone
3 .American Indian alone
4 .Alaska Native alone
5 .American Indian and Alaska Native tribes specified; or
.American Indian or Alaska Native, not specified and no other
.races
6 .Asian alone
7 .Native Hawaiian and Other Pacific Islander alone
8 .Some Other Race alone
9 .Two or More Races

99 Latino/Hispanic (injected in unify_pums)
"""
# RACE_ETHNICITY recoded detailed race (source: the commented-out WHITE_PCT/ASIAN_PCT/PI_PCT/
# BLACK_PCT/INDIAN_PCT rows above) with Latino has a separate category.

RACE_COLUMNS = {
    1: "Proportion of people White",
    2: "Proportion of people Black",
    3: "Proportion of people Indian",
    4: "Proportion of people Indian",
    5: "Proportion of people Indian",
    6: "Proportion of people AAPI",
    7: "Proportion of people AAPI",
    99: "Proportion of people Latino",
}

In [14]:
# area-level companion to the NAICS_{group} dummies above: for each PUMA/MIGPUMA, the
# share of all jobs that fall into each of the 6 skill/credential groups (sum of the
# group's raw NAICS_SECTOR_COLUMNS job counts, divided by total jobs). Used below to
# build OWN_JOB/OWN_JOB_ORIG as "the share of jobs at this location in my own group"
# rather than a raw job count in my narrow individual NAICS sector.
NAICS_GROUP_PCT_COLUMNS = [f"NAICS_GROUP_PCT_{group}" for group in NAICS_GROUPS]


def add_naics_group_pct_columns(census_df):
    total_jobs = census_df[CENSUS_FIELDS["TOT_JOBS"]]
    for group, sectors in NAICS_GROUPS.items():
        group_jobs = census_df[[NAICS_SECTOR_COLUMNS[s] for s in sectors]].sum(axis=1)
        census_df[f"NAICS_GROUP_PCT_{group}"] = np.where(
            total_jobs == 0, 0, group_jobs / total_jobs
        )
    return census_df


puma_census = add_naics_group_pct_columns(puma_census)
migpuma_census = add_naics_group_pct_columns(migpuma_census)

/tmp/ipykernel_568594/92800145.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  census_df[f"NAICS_GROUP_PCT_{group}"] = np.where(
/tmp/ipykernel_568594/92800145.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  census_df[f"NAICS_GROUP_PCT_{group}"] = np.where(
/tmp/ipykernel_568594/92800145.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de

In [15]:
# fill each alternative with census (ACS+LODES), CBSA, distance/time, and NAICS-group
# job-share data. ORIGIN varies per row and the alternative PUMA varies per row too, so
# distance/time lookups are a paired (diagonal) gather -- get positional indices and
# index the underlying numpy arrays directly (DataFrame.lookup() was removed in
# pandas 2.x).
puma_cols = list(CENSUS_FIELDS.values())
alt_col_names = list(CENSUS_FIELDS.keys())

origin_row_pos = distance_matrix.index.get_indexer(df["ORIGIN"])
assert (origin_row_pos >= 0).all(), "some ORIGIN missing from the distance matrix"

# origin (MIGPUMA)-side share of jobs in each NAICS skill/credential group -- a plain
# per-row lookup by ORIGIN, one column per group (rather than gathering just the
# person's own group), so the modeling notebook can weight each group by its own
# coefficient and the person's own NAICS_{group} dummy directly.
origin_group_pct_block = migpuma_census.loc[
    df["ORIGIN"], NAICS_GROUP_PCT_COLUMNS
].reset_index(drop=True)
for col in NAICS_GROUP_PCT_COLUMNS:
    df[col + ".ORIG"] = origin_group_pct_block[col].astype(np.float32).to_numpy()

# person's own age-bracket census column name (constant across alternatives), using the
# AGE_UNDER_18/AGE_18_34/AGE_35_64/AGE_OVER_65 dummies already computed on df (by
# unify_pums.ipynb) rather than re-deriving thresholds from AGEP here
assert (
    df[["AGE_UNDER_18", "AGE_18_34", "AGE_35_64", "AGE_OVER_65"]].sum(axis=1) == 1
).all(), "AGE_* dummies on df aren't mutually exclusive/exhaustive"
own_age_col_name = np.select(
    [
        df["AGE_UNDER_18"] == 1,
        df["AGE_18_34"] == 1,
        df["AGE_35_64"] == 1,
        df["AGE_OVER_65"] == 1,
    ],
    AGE_BRACKET_COLS,
    default="",  # unreachable given the exhaustiveness assert above
)

# person's own RACE_ETHNICITY race census column name (constant across alternatives). RACE_ETHNICITY 8/9
# have no matching proportion field (own_race_missing), so use an arbitrary valid placeholder column
# for the lookup itself and zero out the result afterward -- "placeholder now, mask
# later".
own_race_col_name = df["RACE_ETHNICITY"].map(RACE_COLUMNS).to_numpy()
own_race_missing = pd.isna(own_race_col_name)
own_race_col_name = np.where(
    own_race_missing, next(iter(RACE_COLUMNS.values())), own_race_col_name
)

demo_cols = list(AGE_BRACKET_COLS) + list(dict.fromkeys(RACE_COLUMNS.values()))

dm_np = distance_matrix.to_numpy()
# tm_np = time_matrix.to_numpy()

# NOTE: the recurring pattern here is first getting all the relevant columns for all requested
# alt pumas, storing it into *block
# then, an indexer is created that tells for each person, which index is relevant to them (e.g., correct NAICS gruop)
# afterwards, the block is indexed into by pairs of indices saying that for each person, you should use the corresponding relevant index
alt_blocks = []
for i in range(num_alternatives):
    alt_pumas_i = random_pumas[
        :, i
    ]  # this alternative's destination PUMA, one per person
    key = f"ALT{i + 1}_"

    # curated ACS/LODES fields (CENSUS_FIELDS) for alternative i's PUMA, one row per
    # person -- same value repeats for people who happened to draw the same alt PUMA
    census_block = (
        puma_census.loc[alt_pumas_i, puma_cols]
        .reset_index(drop=True)
        .astype(np.float32)
    )
    census_block.columns = [key + c for c in alt_col_names]
    alt_blocks.append(census_block)

    # CBSA type/name codes for alternative i's PUMA
    cbsa_block = (
        puma_cbsa.loc[alt_pumas_i, ["TYPE_NUM", "NAME_NUM"]]
        .reset_index(drop=True)
        .astype(np.int16)
    )
    cbsa_block.columns = [key + "TYPE", key + "CBSA"]
    alt_blocks.append(cbsa_block)

    # distance/time from each person's own ORIGIN to alternative i's PUMA -- a paired
    # (row, col) gather since both endpoints vary per person, not a simple column select
    col_pos = distance_matrix.columns.get_indexer(alt_pumas_i)
    assert (col_pos >= 0).all(), (
        f"alternative {i} has PUMAs missing from the distance matrix"
    )
    dist = dm_np[origin_row_pos, col_pos]
    # tme = tm_np[origin_row_pos, col_pos]
    alt_blocks.append(
        pd.DataFrame(
            {
                key + "DIST": dist,
                # key + "TIME": tme
            },
            dtype=np.float32,
        )
    )

    # NAICS_GROUP_PCT_*: share of jobs in alternative i's PUMA that fall into each of
    # the 6 skill/credential groups -- one column per group, same idea as census_block
    # above (a plain per-alternative column select+rename, not a per-row gather)
    group_pct_block = (
        puma_census.loc[alt_pumas_i, NAICS_GROUP_PCT_COLUMNS]
        .reset_index(drop=True)
        .astype(np.float32)
    )
    group_pct_block.columns = [key + c for c in NAICS_GROUP_PCT_COLUMNS]
    alt_blocks.append(group_pct_block)

    # OWN_AGE_PCT / OWN_RACE_PCT: per-row gather against the age-bracket and race-share
    # columns -- i.e. "what share of alternative i's population is in this person's own
    # age bracket / race category" (see the pattern blurb above)
    demo_block = puma_census.loc[alt_pumas_i, demo_cols].reset_index(drop=True)
    demo_np = demo_block.to_numpy()

    age_col_pos = demo_block.columns.get_indexer(own_age_col_name)
    own_age_pct = demo_np[np.arange(n), age_col_pos].astype(np.float32)

    race_col_pos = demo_block.columns.get_indexer(own_race_col_name)
    own_race_pct = demo_np[np.arange(n), race_col_pos].astype(np.float32)
    # RACE_ETHNICITY 8/9 (own_race_missing) have no matching census field -- 0 rather than NaN
    own_race_pct = np.where(own_race_missing, 0, own_race_pct).astype(np.float32)

    alt_blocks.append(
        pd.DataFrame(
            {key + "OWN_AGE_PCT": own_age_pct, key + "OWN_RACE_PCT": own_race_pct}
        )
    )

    # the alternative's raw PUMA code itself, so the choice set is recoverable later
    alt_blocks.append(pd.DataFrame({key + "PUMA": alt_pumas_i}))

    # state FIPS code of alternative i's PUMA (guidance: unify_pums.ipynb's commented-out
    # "states" block), sourced from the PUMA<->MIGPUMA equivalency table
    # the state coding should be equivalent to the PUMS one
    alt_blocks.append(
        pd.DataFrame({key + "STATE": puma_migpuma.loc[alt_pumas_i, "State"].to_numpy()})
    )

len(alt_blocks)

1400

In [16]:
# concat the base dataframe with all alternative blocks once (not per-iteration)
df_final = pd.concat([df.reset_index(drop=True)] + alt_blocks, axis=1)
df_final.shape

(253090, 8688)

In [17]:
nas = df_final.isna().sum()
for key in dict(nas[nas > 0]):
    print(key)

CITWP
COW
DRAT
DRATX
ENG
FER
GCL
GCM
GCR
JWMNP
JWRIP
JWTR
MARHD
MARHM
MARHT
MARHW
MARHYP
MLPA
MLPB
MLPCD
MLPE
MLPFG
MLPH
MLPI
MLPJ
MLPK
SCHG
WKHP
WKW
WRK
YOEP
DECADE
DRIVESP
ESP
FOD1P
FOD2P
INDP
JWAP
JWDP
LANP
NAICSP
NOP
OC
OCCP
PAOC
POVPIP
POWPUMA
POWSP
RC
SCIENGP
SCIENGRLP
SFN
SFR
VPS
ACR
MRGP
MRGT
TEN
VALP
VEH
FES
FINCP
FPARC
GRNTP
GRPIP
HHT
HINCP
HUPAOC
HUPARC
MULTG
MV
NOC
OCPIP
PARTNER
R18
R65
SMOCP
TAXAMT
WIF
WKEXREL
WORKSTAT
CBSA_NAME.ORIG
NAICS


In [18]:
df_final.dropna(axis=1, inplace=True)

In [19]:
for col in df_final.columns:
    print(col)

RT
SERIALNO
DIVISION
SPORDER
PUMA
REGION
ST
ADJINC
AGEP
CIT
DDRS
DEAR
DEYE
DOUT
DPHY
DREM
HINS1
HINS2
HINS3
HINS4
HINS5
HINS6
HINS7
INTP
LANX
MAR
MIG
MIL
NWAB
NWAV
NWLA
NWLK
NWRE
OIP
PAP
RELP
RETP
SCH
SCHL
SEMP
SEX
SSIP
SSP
WAGP
WKL
ANC
ANC1P
ANC2P
DIS
ESR
HICOV
HISP
MIGPUMA
MIGSP
MSP
NATIVITY
PERNP
PINCP
POBP
PRIVCOV
PUBCOV
QTRBIR
RAC1P
RAC2P
RAC3P
RACAIAN
RACASN
RACBLK
RACNH
RACNUM
RACPI
RACSOR
RACWHT
WAOB
FAGEP
FCITWP
FFODP
FHISP
FINDP
FINTP
FJWDP
FJWMNP
FJWRIP
FLANP
FMARHYP
FMIGSP
FMILPP
FMILSP
FOCCP
FOIP
FPAP
FPERNP
FPINCP
FPOBP
FPOWSP
FRACP
FRELP
FRETP
FSEMP
FSSIP
FSSP
FWAGP
FWKHP
FYOEP
STAY
ORIGIN
CHOSEN
NP
TYPE
CHILD_UNDER_6
CHILD_6_TO_17
CHILD
WORK2_MAR
WORK1_MAR
SINGLE_PARENT
EDU_NOHIGH
EDU_HIGH
EDU_BACHELORS
AGE_UNDER_18
AGE_18_34
AGE_35_64
AGE_18_22
AGE_23_29
AGE_30_39
AGE_40_49
AGE_50_64
AGE_OVER_65
FOREIGN
IN_COLLEGE
WOMAN_WITH_CHILD
MALE
FEMALE
MARRIED
RECENTLY_WIDOWED_OR_DIVORCED
RECENTLY_MARRIED
MARRIED_MORE_THAN_YEAR
IN_MILITARY
UNEMPLOYED
NOT_IN_LABOR_FORCE
IN_LABOR_

In [20]:
# downcast any purely-integral float columns to save memory -- many census/job-count
# columns are stored as float but every value happens to be a whole number (population
# counts, job counts, etc.), unlike genuine proportions/rates/medians which stay float
float_cols = df_final.columns[df_final.dtypes.isin([np.float32, np.float64])]
downcasts = {}
for col in float_cols:
    if not (df_final[col] % 1 == 0).all():
        continue
    has_na = df_final[col].isna().any()
    assert not has_na  # all nas were dropped earlier

    max_abs = df_final[col].abs().max()
    if max_abs <= np.iinfo(np.int16).max:
        dtype = np.int16
    elif max_abs <= np.iinfo(np.int32).max:
        dtype = np.int32
    else:
        dtype = np.int64
    df_final[col] = df_final[col].astype(dtype)
    downcasts[col] = dtype


df_final = df_final.astype(downcasts)  # single consolidated conversion

for key, typ in downcasts.items():
    print(f"{key} changed to {typ}")

In [ ]:
df_final.to_parquet(f"us_estdata_{year}.parquet")